# Plumbline — lane 6 fine-tune (Colab free T4)

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on 400 hard-negative pairs mined
from the **train split only**, then pushes the checkpoint to the Hugging Face Hub.

Runtime → Change runtime type → **T4 GPU** before running anything.

The hyperparameters are pre-registered and frozen in `docs/02_EVALUATION_SPEC.md` §3.
Do not change them here. A checkpoint trained with different settings must not be
reported against the test split without a new pre-registration.

Expect roughly two minutes of training. If it takes an hour, the runtime is on CPU.

## 1. Confirm the GPU

If this prints `cpu`, stop and switch the runtime type. Training on a Colab CPU
works but is slow, and `EVALUATION_SPEC` §3 says not to.

In [ ]:
import torch
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("torch:", torch.__version__)

## 2. Dependencies

Colab ships torch and transformers already. Only the Hub client is topped up, so
nothing here fights Colab's preinstalled CUDA build of torch.

Deliberately *not* pinned to `requirements.txt`. This is the one step of the
pipeline that does not run on the pinned environment, and the thing that has to be
stable is the **checkpoint format** — a plain HF sequence-classification model —
not the library that produced it. The versions actually used are recorded in
`training_metrics.json` and on the model card.

In [ ]:
!pip install -q -U "huggingface_hub>=0.25" "transformers>=4.40"
import transformers, huggingface_hub
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

## 3. Upload five files

Run the cell, then pick all five at once in the file dialog:

| File | Where it comes from |
|---|---|
| `train_pairs.jsonl` | `data/` — produced by `python -m training.mine_negatives` |
| `mining_report.json` | `data/` — same command; the model card is built from it |
| `train_reranker.py` | `training/` |
| `model_card.md.tmpl` | `training/` |
| `model_card_data.md.tmpl` | `training/` |

All five must land in `/content` (the default). `train_reranker.py` finds the two
templates next to itself, so keep them together.

If `mining_report.json` is missing the run still trains, but the model card loses
its training-data section — including the contamination diagnostic, which is the
part worth publishing.

In [ ]:
from google.colab import files
uploaded = files.upload()
print()
for name in sorted(uploaded):
    print(f"{name:24} {len(uploaded[name]):>9,} bytes")

In [ ]:
# Sanity-check the pair file before spending a GPU on it.
import json, collections

rows = [json.loads(line) for line in open("train_pairs.jsonl", encoding="utf-8") if line.strip()]
labels = collections.Counter(r["label"] for r in rows)
qids = {r["qid"] for r in rows}
print(f"pairs      {len(rows)}")
print(f"questions  {len(qids)}")
print(f"labels     {dict(labels)}   (expect 80 positive, 320 negative)")
print(f"per q      {len(rows) / len(qids):.2f}")
assert labels[1.0] == len(qids), "expected exactly one positive per question"
assert all(r["text"].strip() for r in rows), "a pair has empty chunk text"
print("\nok")

## 4. Hugging Face login

Needs a token with **write** access: huggingface.co/settings/tokens.

`add_to_git_credential=False` because nothing here uses git — the push goes
through the HTTP API.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Train and push

Frozen settings, taken from the module's defaults: 3 epochs, batch 16, lr 2e-5,
10% warmup, BCEWithLogitsLoss, max_length 512, seed 42.

Eight of the 80 questions are held out of *train* as a monitor. It reports
per-epoch loss and `positive_ranked_first` — the fraction of held-out questions
whose positive outscores all four of its own mined negatives. It **does not select
the checkpoint**: eight questions is far too few to pick an epoch on, so the
published checkpoint is always the final one.

Change `--push-to` if you are publishing under a different account.

In [ ]:
!python train_reranker.py \
    --pairs train_pairs.jsonl \
    --mining-report mining_report.json \
    --out ./reranker-v1 \
    --metrics ./training_metrics.json \
    --push-to talalmohsin-98/plumbline-reranker-v1

## 6. Bring the metrics back to the repo

`training_metrics.json` is the provenance for the model card's training table and
belongs in `data/`. Download it and commit it next to `mining_report.json`.

In [ ]:
from google.colab import files
files.download("training_metrics.json")

## 7. Verify the checkpoint loads the way lane 4's does

The whole comparison rests on the two checkpoints being the same *kind* of
artefact. This prints `num_labels` and the activation `CrossEncoder` picks for
each. **They must match.** If they do not, lane 6 differs from lane 4 in its
scoring convention as well as its weights, and the delta is not attributable to
the fine-tune.

In [ ]:
!pip install -q sentence-transformers
from sentence_transformers import CrossEncoder

stock = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)
tuned = CrossEncoder("talalmohsin-98/plumbline-reranker-v1", max_length=512)

for name, m in (("stock", stock), ("tuned", tuned)):
    act = getattr(m, "default_activation_function", None) or getattr(m, "activation_fn", None)
    print(f"{name:6} num_labels={m.model.config.num_labels}  activation={type(act).__name__}")

pair = [("which package is needed for form data?",
         "To receive form data, first install python-multipart.")]
print("\nstock:", stock.predict(pair))
print("tuned:", tuned.predict(pair))

## 8. Back in the repo

```bash
python -m backend.evaluate --split test    # now scores lanes 1-6
python -m backend.significance             # McNemar + paired bootstrap vs lane 4
```

**Read the tripwire before reading the result.** Lane 6 above 0.95 recall@10
(≥ 34/35) means test questions reached training. Stop, audit `data/train_pairs.jsonl`
against `data/test.jsonl`, and do not report the number until the audit is done.

And the pre-registered rule that matters most: **if lane 6 loses to lane 4, that is
the published result.** Do not come back to this notebook and change a
hyperparameter.